# General tracking-scene visualizer

The generic scene picker remains reusable. This notebook composes it with the
latest Stage 8 paths, ordinary cell centers, reconstructed merge centers,
cell-ID labels, coordinate-mode alignment, and original-frame display.


In [ ]:
%gui qt


In [ ]:
from importlib import import_module

from src.io import PipelinePaths

paths = PipelinePaths.discover()
SCENES_ROOT = paths.tracking_scenes
TRACKS_PATH = paths.stage8_stitching / "tracks.csv"
TRACKING_METADATA_PATH = paths.stage8_stitching / "metadata.json"
USE_ORIGINAL_COORDINATES = False
SHOW_STAGE_8_OVERLAY = True

overlay_module = import_module("src.09_visualization.step04_scene_overlay")
prepare_stage8_scene_overlay = overlay_module.prepare_stage8_scene_overlay


In [ ]:
from pathlib import Path
from typing import Any

import napari
from qtpy.QtWidgets import QCheckBox, QGroupBox, QLabel, QVBoxLayout

from diagnostics.tracking_scene_extraction.napari_scene_visualizer import (
    TrackingSceneVisualizerWidget,
    add_scene_to_viewer,
    load_tracking_scene,
)

STAGE_8_LAYER_PREFIX = "Stage 8 | "


def remove_stage8_layers(viewer: Any) -> None:
    for layer in list(viewer.layers):
        if str(layer.name).startswith(STAGE_8_LAYER_PREFIX):
            viewer.layers.remove(layer)


def add_stage8_layers(viewer: Any, overlay) -> dict[str, Any]:
    remove_stage8_layers(viewer)
    layers = {
        "tracks": viewer.add_tracks(
            overlay.track_data,
            name=f"{STAGE_8_LAYER_PREFIX}Tracks",
            scale=overlay.scale,
            translate=overlay.translate,
            tail_length=max(int(overlay.summary["scene_frame_count"]) + 2, 2),
            tail_width=3,
        )
    }
    if len(overlay.ordinary_point_data):
        layers["centers"] = viewer.add_points(
            overlay.ordinary_point_data,
            name=f"{STAGE_8_LAYER_PREFIX}Cell centers (C=cell ID)",
            scale=overlay.scale,
            translate=overlay.translate,
            size=5,
            face_color="red",
            properties=overlay.ordinary_properties,
            text={
                "string": "{display_label}",
                "size": 8,
                "color": "white",
                "anchor": "upper_left",
            },
        )
    if len(overlay.virtual_point_data):
        layers["virtual_centers"] = viewer.add_points(
            overlay.virtual_point_data,
            name=f"{STAGE_8_LAYER_PREFIX}Virtual merge centers",
            scale=overlay.scale,
            translate=overlay.translate,
            size=9,
            face_color="yellow",
            properties=overlay.virtual_properties,
            text={
                "string": "{virtual_display_label}",
                "size": 9,
                "color": "white",
                "anchor": "upper_left",
            },
        )
    return layers


class GeneralTrackingSceneVisualizerWidget(TrackingSceneVisualizerWidget):
    # Generic scene picker composed with an optional latest Stage 8 overlay.

    def __init__(
        self,
        *,
        viewer: Any,
        scenes_root: str | Path,
        tracks_path: str | Path,
        metadata_path: str | Path,
        use_original_coordinates: bool = False,
        show_stage8_overlay: bool = True,
    ) -> None:
        self.tracks_path = Path(tracks_path)
        self.metadata_path = Path(metadata_path)
        self._initial_show_stage8_overlay = bool(show_stage8_overlay)
        super().__init__(
            viewer=viewer,
            scenes_root=scenes_root,
            use_original_coordinates=use_original_coordinates,
        )

    def _build_ui(self, use_original_coordinates: bool) -> None:
        super()._build_ui(use_original_coordinates)
        overlay_group = QGroupBox("Latest tracking result")
        overlay_layout = QVBoxLayout()
        overlay_group.setLayout(overlay_layout)
        self.stage8_checkbox = QCheckBox("Show latest Stage 8 paths and centers")
        self.stage8_checkbox.setChecked(self._initial_show_stage8_overlay)
        overlay_layout.addWidget(self.stage8_checkbox)
        self.overlay_status_label = QLabel("Load a scene to resolve its tracks.")
        self.overlay_status_label.setWordWrap(True)
        overlay_layout.addWidget(self.overlay_status_label)
        root_layout = self.layout()
        root_layout.insertWidget(root_layout.count() - 1, overlay_group)

    def _connect_events(self) -> None:
        super()._connect_events()
        self.stage8_checkbox.toggled.connect(self._on_stage8_toggled)
        self.viewer.dims.events.current_step.connect(self._on_viewer_step_changed)

    def _on_viewer_step_changed(self, _event: Any | None = None) -> None:
        if self.current_scene is not None:
            self._update_info(self.current_scene)

    def _current_scene_frames(self, scene) -> tuple[str, str]:
        if len(scene.frames) == 0:
            return "-", "-"
        try:
            local_time = int(round(float(self.viewer.dims.current_step[0])))
        except Exception:
            return "unavailable", "unavailable"
        if not 0 <= local_time < len(scene.frames):
            return str(local_time), "outside saved scene"
        return str(local_time), str(int(scene.frames[local_time]))

    def _update_info(self, scene) -> None:
        selected_cells = scene.metadata.get("selected_cells", {})
        selected_count = sum(len(values) for values in selected_cells.values())
        first_frame = int(scene.frames[0]) if len(scene.frames) else "-"
        last_frame = int(scene.frames[-1]) if len(scene.frames) else "-"
        shape = tuple(int(value) for value in scene.instance_labels.shape[1:])
        local_time, original_frame = self._current_scene_frames(scene)
        self.info_label.setText(
            f"Category: {scene.category}\n"
            f"Scene: {scene.name}\n"
            f"Sample: {scene.sample_id}\n"
            f"Current scene time index: {local_time}\n"
            f"Current original frame: {original_frame}\n"
            f"Saved frames: {first_frame}–{last_frame} ({len(scene.frames)})\n"
            f"Selected cell entries: {selected_count}\n"
            f"Crop shape ZYX: {shape}\n"
            f"Crop origin ZYX: {scene.crop_origin_zyx}\n"
            f"Voxel size ZYX: {scene.voxel_size_zyx}"
        )

    def load_selected_scene(self) -> None:
        try:
            scene = load_tracking_scene(self.selected_scene_path())
            add_scene_to_viewer(
                self.viewer,
                scene,
                use_original_coordinates=self.original_coordinates_checkbox.isChecked(),
                remove_existing=True,
            )
        except Exception as error:
            self._show_error(str(error))
            return
        self.current_scene = scene
        self._update_info(scene)
        self.status_label.setText(
            f"Loaded {scene.category}/{scene.name}. {self._refresh_stage8_overlay()}"
        )

    def _reload_current_scene(self, _checked: bool) -> None:
        if self.current_scene is None:
            return
        try:
            add_scene_to_viewer(
                self.viewer,
                self.current_scene,
                use_original_coordinates=self.original_coordinates_checkbox.isChecked(),
                remove_existing=True,
            )
            self._update_info(self.current_scene)
            message = self._refresh_stage8_overlay()
            self.status_label.setText(message)
        except Exception as error:
            self._show_error(str(error))

    def _on_stage8_toggled(self, checked: bool) -> None:
        if not checked:
            remove_stage8_layers(self.viewer)
            self.overlay_status_label.setText("Latest Stage 8 overlay is disabled.")
        elif self.current_scene is not None:
            self.status_label.setText(self._refresh_stage8_overlay())

    def _refresh_stage8_overlay(self) -> str:
        remove_stage8_layers(self.viewer)
        if self.current_scene is None:
            message = "No scene is loaded."
            self.overlay_status_label.setText(message)
            return message
        if not self.stage8_checkbox.isChecked():
            message = "Latest Stage 8 overlay is disabled."
            self.overlay_status_label.setText(message)
            return message
        try:
            overlay = prepare_stage8_scene_overlay(
                self.current_scene,
                tracks_path=self.tracks_path,
                metadata_path=self.metadata_path,
                use_original_coordinates=self.original_coordinates_checkbox.isChecked(),
            )
            add_stage8_layers(self.viewer, overlay)
        except Exception as error:
            message = f"Stage 8 overlay unavailable: {error}"
            self.overlay_status_label.setText(message)
            return message
        track_ids = ", ".join(str(value) for value in overlay.summary["track_ids"])
        message = (
            f"Stage 8 overlay loaded: {overlay.summary['track_row_count']} rows, "
            f"tracks [{track_ids}], "
            f"{overlay.summary['virtual_center_count']} virtual center(s)."
        )
        self.overlay_status_label.setText(message)
        return message

    def clear_scene(self) -> None:
        remove_stage8_layers(self.viewer)
        super().clear_scene()
        self.overlay_status_label.setText("Load a scene to resolve its tracks.")


viewer = napari.Viewer(ndisplay=3, title="General Tracking Scene Visualizer")
scene_browser = GeneralTrackingSceneVisualizerWidget(
    viewer=viewer,
    scenes_root=SCENES_ROOT,
    tracks_path=TRACKS_PATH,
    metadata_path=TRACKING_METADATA_PATH,
    use_original_coordinates=USE_ORIGINAL_COORDINATES,
    show_stage8_overlay=SHOW_STAGE_8_OVERLAY,
)
viewer.window.add_dock_widget(
    scene_browser,
    area="right",
    name="General Tracking Scene Visualizer",
)
viewer
